# Практика 6 Интерактивная сегментация изображений и видео


## 0. Среда и данные

Используем зафиксированный commit SAM 2 и `sam2.1_hiera_small`

In [ ]:
import os, sys, subprocess, time, gc, shutil
from pathlib import Path
from urllib.request import Request, urlopen

SAM2_COMMIT = "2b90b9f5ceec907a1c18123530e92e794ad901a4"
SAM2_ROOT = Path("/content/sam2") if Path("/content").exists() else Path("sam2")
CHECKPOINT = Path("/content/sam2.1_hiera_small.pt") if Path("/content").exists() else Path("sam2.1_hiera_small.pt")

def run(command, **kwargs):
    # Print and execute a command; a non-zero return code stops the notebook.
    print("$", " ".join(map(str, command)))
    subprocess.run(command, check=True, **kwargs)

if not SAM2_ROOT.exists():
    run(["git", "clone", "--quiet", "https://github.com/facebookresearch/sam2.git", str(SAM2_ROOT)])
run(["git", "-C", str(SAM2_ROOT), "checkout", "--quiet", SAM2_COMMIT])

install_env = os.environ.copy()
install_env["SAM2_BUILD_CUDA"] = "0"
run([sys.executable, "-m", "pip", "install", "-q", "-e", str(SAM2_ROOT),
     "imageio[ffmpeg]", "scipy", "progressbar2", "hickle"], env=install_env)

if not CHECKPOINT.exists():
    run(["curl", "-L", "--fail", "--retry", "3", "-o", str(CHECKPOINT),
         "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_small.pt"])

import torch
assert torch.cuda.is_available(), "Выберите Runtime → Change runtime type → T4 GPU"
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("SAM 2 checkpoint:", round(CHECKPOINT.stat().st_size / 2**20, 1), "MB")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Video, display

plt.rcParams.update({"figure.figsize": (12, 5), "axes.grid": False})

DATA_ROOT = Path("/content/DAVIS-mini") if Path("/content").exists() else Path("data/DAVIS-mini")
HF_COMMIT = "3c33c3245ce366cf3f533d35f4a3f1c295ab889c"
HF_BASE = f"https://huggingface.co/datasets/AlonzoLeeeooo/DAVIS-Edit/resolve/{HF_COMMIT}"
SEQUENCE_LENGTHS = {"bmx-trees": 80, "libby": 49}

def download_file(url, destination, retries=3):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists():
        return
    request = Request(url, headers={"User-Agent": "video-segmentation-practice/2.0"})
    for attempt in range(retries):
        try:
            destination.write_bytes(urlopen(request, timeout=60).read())
            return
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(attempt + 1)

for sequence, length in SEQUENCE_LENGTHS.items():
    for index in range(length):
        stem = f"{index:05d}"
        download_file(f"{HF_BASE}/JPEGImages/{sequence}/{stem}.jpg",
                      DATA_ROOT / "JPEGImages" / "480p" / sequence / f"{stem}.jpg")
        download_file(f"{HF_BASE}/Annotations/{sequence}/{stem}.png",
                      DATA_ROOT / "Annotations" / "480p" / sequence / f"{stem}.png")

def load_sequence(sequence):
    frame_dir = DATA_ROOT / "JPEGImages" / "480p" / sequence
    mask_dir = DATA_ROOT / "Annotations" / "480p" / sequence
    frame_paths = sorted(frame_dir.glob("*.jpg"))
    mask_paths = sorted(mask_dir.glob("*.png"))
    frames = [np.asarray(Image.open(path).convert("RGB")) for path in frame_paths]
    # В используемом mirror один объект закодирован значениями 0/255.
    masks = [np.asarray(Image.open(path)) > 0 for path in mask_paths]
    assert len(frames) == len(masks) == SEQUENCE_LENGTHS[sequence]
    assert all(frame.shape[:2] == mask.shape for frame, mask in zip(frames, masks))
    return frame_dir, mask_dir, frames, masks

FRAME_DIR, MASK_DIR, frames, gts = load_sequence("bmx-trees")
LIBBY_FRAME_DIR, LIBBY_MASK_DIR, libby_frames, libby_gts = load_sequence("libby")
N_FRAMES = len(frames)
print("bmx-trees:", len(frames), frames[0].shape, "libby:", len(libby_frames), libby_frames[0].shape)

In [ ]:
def overlay(frame, mask, color=(0, 220, 255), alpha=.55):
    # Blend a binary mask over an RGB image.
    result = frame.astype(np.float32).copy()
    region = np.asarray(mask, bool)
    result[region] = (1-alpha)*result[region] + alpha*np.asarray(color)
    return np.clip(result, 0, 255).astype(np.uint8)

def show_frames(frame_list, indices, masks=None, title=""):
    columns = 4
    rows = int(np.ceil(len(indices) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(16, 3.6*rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, index in zip(axes.flat, indices):
        image = frame_list[index] if masks is None else overlay(frame_list[index], masks[index])
        ax.imshow(image); ax.set_title(f"кадр {index}"); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

def save_video(path, images, fps=12):
    import imageio.v2 as imageio
    imageio.mimsave(path, images, fps=fps, macro_block_size=2)
    return Path(path)

KEY_FRAMES = [0, 10, 20, 30, 40, 50, 60, 79]
show_frames(frames, KEY_FRAMES, title="bmx-trees: предскажите участок сбоя")
show_frames(frames, KEY_FRAMES, gts, title="GT: человек + велосипед — один целевой объект")
show_frames(libby_frames, [0, 8, 16, 24, 32, 40, 48], title="libby: второй тип движения")

In [ ]:
LIBBY_KEY_FRAMES = [0, 8, 16, 24, 32, 40, 48]
show_frames(libby_frames, LIBBY_KEY_FRAMES, libby_gts,
            title="libby: GT-маска собаки")

SWAN_LENGTH = 50
for index in range(SWAN_LENGTH):
    stem = f"{index:05d}"
    download_file(f"{HF_BASE}/JPEGImages/blackswan/{stem}.jpg",
                  DATA_ROOT/"JPEGImages"/"480p"/"blackswan"/f"{stem}.jpg")
    download_file(f"{HF_BASE}/Annotations/blackswan/{stem}.png",
                  DATA_ROOT/"Annotations"/"480p"/"blackswan"/f"{stem}.png")

SEQUENCE_LENGTHS["blackswan"] = SWAN_LENGTH
SWAN_FRAME_DIR, SWAN_MASK_DIR, swan_frames, swan_gts = load_sequence("blackswan")
SWAN_KEY_FRAMES = [0, 7, 14, 21, 28, 35, 42, 49]
show_frames(swan_frames, SWAN_KEY_FRAMES, title="blackswan: исходные кадры")
show_frames(swan_frames, SWAN_KEY_FRAMES, swan_gts, title="blackswan: GT-маска")

## 1. SAM 2 на отдельных изображениях

SAM 2 — **promptable segmentation model**: вместо фиксированного списка классов она получает изображение
и подсказку пользователя, а на выходе строит маску указанного объекта. Поэтому модель не «ищет
велосипедиста» как класс. Она отвечает на более общий вопрос: *какую область пользователь имел в виду,
указав эти точки или рамку?*

### Основные блоки

1. **Image encoder (Hiera)** один раз превращает изображение в многомасштабные признаки. Это самая
   дорогая часть, поэтому `set_image` кеширует результат.
2. **Prompt encoder** кодирует точки, box или входную маску. Он различает положительные и отрицательные
   точки и сохраняет их координаты.
3. **Mask decoder** объединяет признаки изображения и промпта, строит маску и оценивает её качество.
4. Для видео добавляются **memory attention**, **memory encoder** и **memory bank**. К ним вернёмся в
   разделе 3; на одиночной картинке временная память не нужна.

### Что означают клики

- **Положительный клик (`label=1`)**: «эта точка обязана находиться внутри нужного объекта».
- **Отрицательный клик (`label=0`)**: «эту точку маска захватывать не должна».
- Клик **не является названием класса** и не задаёт масштаб. Точка на шортах совместима и с маской
  шорт, и с человеком, и с композицией «человек + велосипед».
- Несколько разнесённых положительных точек сообщают модели, что удалённые части относятся к одному
  объекту. Отрицательные точки полезны, когда маска захватывает фон или соседний объект.

### Что означают промпты

| Параметр | Смысл |
|---|---|
| `point_coords` | массив `(x, y)` в координатах исходного изображения, не `(row, col)` |
| `point_labels` | `1` — объект, `0` — фон; порядок соответствует координатам |
| `box` | `[x0, y0, x1, y1]`, приблизительная область всего нужного объекта |
| `multimask_output=True` | вернуть три варианта, потому что один короткий промпт часто неоднозначен |
| `predicted_iou` | самооценка маски моделью |


![Архитектура SAM 2](https://raw.githubusercontent.com/facebookresearch/sam2/main/assets/model_diagram.png)

In [ ]:
# Добавляем установленный репозиторий SAM 2 в поиск Python-модулей.
sys.path.insert(0, str(SAM2_ROOT))

# distance_transform_edt понадобится для выбора точки глубоко внутри маски:
# такая точка устойчивее к небольшой ошибке мыши, чем точка у границы.
from scipy.ndimage import distance_transform_edt
from sam2.build_sam import build_sam2, build_sam2_video_predictor
from sam2.sam2_image_predictor import SAM2ImagePredictor

# Конфигурация задаёт размер варианта Hiera. Small.
MODEL_CFG = "configs/sam2.1/sam2.1_hiera_s.yaml"
device = torch.device("cuda")

# build_sam2 создаёт общую модель и загружает checkpoint на GPU.
# SAM2ImagePredictor — удобная оболочка именно для отдельных изображений.
image_model = build_sam2(MODEL_CFG, str(CHECKPOINT), device=device)
image_predictor = SAM2ImagePredictor(image_model)

def median_click(mask):
    '''Точка около координатных медиан foreground-пикселей.

    Для невыпуклой маски медиана может попасть в тонкую или семантически отдельную часть.
    Поэтому после вычисления выбирается ближайший пиксель, который точно лежит внутри mask.
    '''
    ys, xs = np.nonzero(mask)
    cx, cy = np.median(xs), np.median(ys)
    nearest = np.argmin((xs-cx)**2 + (ys-cy)**2)
    return int(xs[nearest]), int(ys[nearest])

def deepest_click(mask):
    '''Foreground-пиксель с максимальным расстоянием до ближайшей границы.'''
    distance = distance_transform_edt(np.asarray(mask, bool))
    y, x = np.unravel_index(np.argmax(distance), distance.shape)
    return int(x), int(y)  # SAM принимает (x, y), тогда как NumPy индексирует [y, x].

def two_spread_clicks(mask):
    '''Две положительные точки в удалённых плотных частях объекта.'''
    first = deepest_click(mask)
    yy, xx = np.indices(mask.shape)
    # Второй score одновременно предпочитает внутренность mask и удаление от первой точки.
    inside = distance_transform_edt(np.asarray(mask, bool))
    separation = np.hypot(xx-first[0], yy-first[1])
    score = inside * separation
    y2, x2 = np.unravel_index(np.argmax(score), score.shape)
    return [first, (int(x2), int(y2))]

def bounding_box(mask, margin=6):
    '''Box вокруг foreground с небольшим отступом; координаты обрезаются по изображению.'''
    ys, xs = np.nonzero(mask)
    height, width = mask.shape
    return np.asarray([max(0, xs.min()-margin), max(0, ys.min()-margin),
                       min(width-1, xs.max()+margin), min(height-1, ys.max()+margin)], np.float32)

def iou(mask, target):
    '''Истинный region IoU; он используется только для анализа, а не подаётся модели.'''
    union = np.logical_or(mask, target).sum()
    return 1.0 if union == 0 else float(np.logical_and(mask, target).sum()/union)

In [ ]:
def predict_still(frame, *, points=None, labels=None, box=None, multimask=True):
    '''Запускает SAM 2 на одном RGB-кадре и возвращает бинарные маски и predicted IoU.'''
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        # Encoder считается один раз. После set_image можно быстро пробовать разные клики.
        image_predictor.set_image(np.asarray(frame).copy())
        masks, scores, logits = image_predictor.predict(
            # None означает, что соответствующий тип промпта в этом опыте не используется.
            point_coords=None if points is None else np.asarray(points, np.float32),
            point_labels=None if labels is None else np.asarray(labels, np.int32),
            box=None if box is None else np.asarray(box, np.float32),
            # Для короткого неоднозначного промпта просим три варианта масштаба объекта.
            multimask_output=multimask,
        )
    # Порог уже применён внутри predictor; приводим тип к bool для метрик и overlay.
    return masks.astype(bool), scores

# Сравниваем прежнюю координатную эвристику и более безопасную глубокую точку.
old_click = median_click(gts[0])
deep_click = deepest_click(gts[0])
old_masks, old_scores = predict_still(frames[0], points=[old_click], labels=[1])

# Один клик породил три гипотезы. predicted IoU известен модели, true IoU считаем по учебному GT.
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for index, (ax, mask, score) in enumerate(zip(axes, old_masks, old_scores)):
    ax.imshow(overlay(frames[0], mask))
    ax.scatter(*old_click, c="lime", marker="*", s=170)
    ax.set_title(f"hyp {index}: predicted={score:.3f}\ntrue IoU={iou(mask, gts[0]):.3f}")
    ax.axis("off")
plt.suptitle("Один положительный клик допускает разные уровни объекта")
plt.show()

# Oracle-выбор нужен только для анализа возможностей. В реальном приложении GT неизвестен.
best_index = int(np.argmax([iou(mask, gts[0]) for mask in old_masks]))
best_image_mask = old_masks[best_index]
print("median click:", old_click, "deepest click:", deep_click, "best-by-GT hypothesis:", best_index)

### Лаборатория промптов

Теперь меняем только промпт, оставляя изображение и модель неизменными. Так можно отделить влияние
интерфейса разметки от качества backbone.

Проверим семь ситуаций:
1. координатная медиана;
2. глубокий клик;
3. точка в верхней части целевого объекта;
4. точка в нижней части — она чаще провоцирует выбор велосипеда или части одежды;
5. две разнесённые положительные точки;
6. positive + negative click;
7. box, явно задающий пространственный охват.


In [ ]:
# Строим дополнительные точки автоматически, чтобы опыт был воспроизводим.
ys0, xs0 = np.nonzero(gts[0])
middle_y = int(np.median(ys0))
row_grid = np.indices(gts[0].shape)[0]
upper_click = deepest_click(gts[0] & (row_grid <= middle_y))
lower_click = deepest_click(gts[0] & (row_grid > middle_y))
spread_clicks = two_spread_clicks(gts[0])

# Красная точка располагается снаружи GT рядом с правой верхней частью box.
box0 = bounding_box(gts[0])
negative_point = (int(min(frames[0].shape[1]-1, box0[2]+18)), int(box0[1]))

prompt_specs = [
    ("median point", dict(points=[old_click], labels=[1], multimask=True)),
    ("deepest point", dict(points=[deep_click], labels=[1], multimask=True)),
    ("upper point", dict(points=[upper_click], labels=[1], multimask=True)),
    ("lower point", dict(points=[lower_click], labels=[1], multimask=True)),
    ("two positives", dict(points=spread_clicks, labels=[1, 1], multimask=True)),
    ("positive + negative", dict(points=[deep_click, negative_point], labels=[1, 0], multimask=True)),
    ("box", dict(box=box0, multimask=False)),
]

prompt_results = []
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
for ax in axes.flat:
    ax.axis("off")

for ax, (name, kwargs) in zip(axes.flat, prompt_specs):
    masks, scores = predict_still(frames[0], **kwargs)
    true_ious = np.asarray([iou(mask, gts[0]) for mask in masks])

    # model_choice доступен без GT; oracle_choice нужен только для учебного анализа.
    model_choice = int(np.argmax(scores))
    oracle_choice = int(np.argmax(true_ious))
    prompt_results.append({
        "name": name,
        "model_choice": model_choice,
        "oracle_choice": oracle_choice,
        "predicted": float(scores[model_choice]),
        "true_model": float(true_ious[model_choice]),
        "true_oracle": float(true_ious[oracle_choice]),
    })

    # На рисунке показываем честный автоматический выбор по predicted IoU.
    ax.imshow(overlay(frames[0], masks[model_choice]))
    ax.set_title(f"{name}\nmodel hyp={model_choice}, true IoU={true_ious[model_choice]:.3f}")
    if kwargs.get("points"):
        colors = ["lime" if label == 1 else "red" for label in kwargs["labels"]]
        px, py = zip(*kwargs["points"])
        ax.scatter(px, py, c=colors, s=100, marker="*")
    if kwargs.get("box") is not None:
        x0, y0, x1, y1 = kwargs["box"]
        ax.add_patch(plt.Rectangle((x0, y0), x1-x0, y1-y0,
                                   fill=False, color="yellow", lw=2))
    ax.axis("off")
plt.tight_layout(); plt.show()

print(f"{'prompt':20s} {'model hyp':>9s} {'oracle hyp':>10s} {'pred IoU':>9s} {'true/model':>11s} {'true/oracle':>11s}")
for row in prompt_results:
    print(f"{row['name']:20s} {row['model_choice']:9d} {row['oracle_choice']:10d} "
          f"{row['predicted']:9.3f} {row['true_model']:11.3f} {row['true_oracle']:11.3f}")

- Если `model hyp` и `oracle hyp` различаются, проблема не в отсутствии хорошей маски среди трёх, а в
  ранжировании: predicted IoU выбрал не тот пользовательский смысл.
- Разница между upper/lower point демонстрирует семантическую неоднозначность координаты: оба клика
  находятся внутри GT, но активируют разные части сцены.
- Две positive points полезны, когда части объекта удалены друг от друга. Negative point исправляет
  захват конкретного фона, но не обязан решать неоднозначность масштаба.
- Box обычно выигрывает на «человек + велосипед», потому что сообщает полный охват. Здесь box вычислен
  по GT ради повторяемости; в интерфейсе пользователь рисовал бы приблизительную рамку вручную.

### Перенос вывода на вторую сцену

In [ ]:
# Второй пример проверяет, переносится ли вывод на другую сцену.
libby_click = deepest_click(libby_gts[0])
libby_box = bounding_box(libby_gts[0])
libby_point_masks, libby_point_scores = predict_still(
    libby_frames[0], points=[libby_click], labels=[1], multimask=True)
libby_box_masks, libby_box_scores = predict_still(
    libby_frames[0], box=libby_box, multimask=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
variants = [(libby_frames[0], "исходное"),
            (overlay(libby_frames[0], libby_point_masks[np.argmax(libby_point_scores)]), "лучший predicted point"),
            (overlay(libby_frames[0], libby_box_masks[0]), "box")]
for ax, (image, title) in zip(axes, variants):
    ax.imshow(image); ax.set_title(title); ax.axis("off")
plt.show()

## 2. Метрики без магии формул


Пусть $M$ — предсказанная бинарная маска, $G$ — ground truth. Обозначим:

- $TP=|M\cap G|$ — правильно найденные пиксели объекта;
- $FP=|M\setminus G|$ — фон, ошибочно включённый в маску;
- $FN=|G\setminus M|$ — пропущенные пиксели объекта.

### J: совпадение площади

`intersection` — пиксели, где модель и GT согласны; `union` — всё, что отметил хотя бы один из них.

$$J(M,G)=\frac{|M\cap G|}{|M\cup G|}=\frac{TP}{TP+FP+FN}.$$

$J=1$ означает полное совпадение, $J=0$ — отсутствие пересечения. Метрика одинаково штрафует лишнюю
область и пропуск, но не сообщает, *где* проходит ошибка.

### F: совпадение границ

Обозначим границы масок как $\partial M$ и $\partial G$, а $N_\tau(\partial G)$ — полосу ширины
$\tau$ вокруг истинной границы. Небольшой допуск нужен, потому что аннотаторы могут провести контур
на один-два пикселя по-разному.

$$P=\frac{|\partial M\cap N_\tau(\partial G)|}{|\partial M|},\qquad
R=\frac{|\partial G\cap N_\tau(\partial M)|}{|\partial G|},$$

$$F=\frac{2PR}{P+R}.$$

Precision отвечает: «какая доля предсказанной границы находится рядом с правильной?». Recall отвечает:
«какая доля истинной границы найдена?». Гармоническое среднее требует, чтобы хорошими были оба числа.


$$J\&F=\frac{J_{mean}+F_{mean}}{2},$$

где сначала J и F усредняются по оцениваемым кадрам. В нашей практике кадр 0 не входит в среднее,
потому что его маска использована как входной prompt.

Ниже показаны intersection, union, обе границы и tolerance band.

In [ ]:
from scipy.ndimage import binary_dilation, binary_erosion

def jaccard(mask, gt):
    '''Region IoU/Jaccard для двух бинарных масок одинакового размера.'''
    mask = np.asarray(mask, bool)
    gt = np.asarray(gt, bool)

    # OR формирует union, AND — intersection.
    union = np.logical_or(mask, gt).sum()
    intersection = np.logical_and(mask, gt).sum()

    return 1.0 if union == 0 else float(intersection / union)

def mask_boundary(mask):
    '''Внутренняя однопиксельная граница: mask минус её морфологическая эрозия.'''
    mask = np.asarray(mask, bool)
    eroded = binary_erosion(mask, structure=np.ones((3, 3), bool), border_value=0)
    return mask & ~eroded

def boundary_details(mask, gt, bound_th=.008, radius=None):
    '''Возвращает F и все промежуточные изображения для объяснения метрики.'''
    mask, gt = np.asarray(mask, bool), np.asarray(gt, bool)

    # Явно задаём крайние случаи, чтобы не делить на ноль.
    if not mask.any() and not gt.any():
        return dict(F=1., precision=1., recall=1., bm=mask, bg=gt,
                    band_m=mask, band_g=gt)
    if not mask.any() or not gt.any():
        return dict(F=0., precision=0., recall=0., bm=mask, bg=gt,
                    band_m=mask, band_g=gt)

    bm = mask_boundary(mask)  # boundary of model mask
    bg = mask_boundary(gt)    # boundary of ground truth

    # В DAVIS tolerance пропорционален диагонали кадра. Для примера radius задаётся вручную.
    if radius is None:
        radius = max(1, int(np.ceil(bound_th * np.hypot(*mask.shape))))
    window = np.ones((2*radius+1, 2*radius+1), bool)

    # Dilation превращает тонкую границу в допустимую полосу совпадения.
    band_m = binary_dilation(bm, structure=window)
    band_g = binary_dilation(bg, structure=window)
    matched_m = bm & band_g  # точки model boundary, попавшие в полосу GT
    matched_g = bg & band_m  # точки GT boundary, найденные моделью

    precision = matched_m.sum() / bm.sum()
    recall = matched_g.sum() / bg.sum()
    f = 0.0 if precision + recall == 0 else 2*precision*recall/(precision+recall)
    return dict(F=float(f), precision=float(precision), recall=float(recall),
                bm=bm, bg=bg, band_m=band_m, band_g=band_g,
                matched_m=matched_m, matched_g=matched_g, radius=radius)

def boundary_f(mask, gt, bound_th=.008, radius=None):
    '''Короткая обёртка для случаев, когда нужны только итоговые F, а не диагностика.'''
    return boundary_details(mask, gt, bound_th, radius)["F"]

In [ ]:
# Два квадрата 6×6: prediction сдвинут вниз на 1 и вправо на 2 пикселя.
gt_toy = np.zeros((12,12), bool)
gt_toy[3:9, 3:9] = True
pred_toy = np.zeros((12,12), bool)
pred_toy[4:10, 5:11] = True

# Компоненты J можно буквально посчитать по белым пикселям.
intersection = pred_toy & gt_toy
union = pred_toy | gt_toy

# radius=1 означает: контуры на расстоянии не более одного пикселя считаются совпавшими.
details = boundary_details(pred_toy, gt_toy, radius=1)

panels = [gt_toy, pred_toy, intersection, union, details["bg"], details["bm"],
          details["band_g"], details["matched_m"]]
titles = ["GT", "prediction", f"intersection={intersection.sum()}", f"union={union.sum()}",
          "GT boundary", "pred boundary", "GT tolerance band", "matched pred boundary"]
fig, axes = plt.subplots(2,4,figsize=(14,7))
for ax, panel, title in zip(axes.flat, panels, titles):
    ax.imshow(panel, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.set_xticks(range(12)); ax.set_yticks(range(12))
    ax.grid(color="lightgray", lw=.4)
plt.tight_layout(); plt.show()

print(f"J = {intersection.sum()}/{union.sum()} = {jaccard(pred_toy, gt_toy):.3f}")
print(f"boundary precision={details['precision']:.3f}, "
      f"recall={details['recall']:.3f}, F={details['F']:.3f}")

assert np.isclose(jaccard(pred_toy, gt_toy), 20/52)
assert np.isclose(details["F"], .6)

In [ ]:
# Создаём разные виды ошибок относительно одного GT.
perfect = gt_toy.copy()
shifted = np.roll(gt_toy, 2, axis=1)              # правильная форма, неверное положение
too_large = binary_dilation(gt_toy, iterations=2) # захвачен фон вокруг объекта
with_hole = gt_toy.copy(); with_hole[5:7,5:7] = False  # маленькая внутренняя дырка
jagged = gt_toy.copy(); jagged[3:9:2, 3] = False       # несколько дефектов контура

cases = {"perfect": perfect, "shifted": shifted, "too large": too_large,
         "hole": with_hole, "jagged": jagged, "empty": np.zeros_like(gt_toy)}

print(f"{'case':12s} {'J':>6s} {'F':>6s}")
for name, prediction in cases.items():
    j = jaccard(prediction, gt_toy)
    f = boundary_f(prediction, gt_toy, radius=1)
    print(f"{name:12s} {j:6.3f} {f:6.3f}")

- `perfect`: обе метрики равны 1.
- `shifted`: форма сохранена, но область стоит не там — J падает до `0.500`; tolerance оставляет F=`0.600`.
- `too large`: захвачено много фона, поэтому низки и J, и F.
- `hole` и `jagged`: J замечает изменившуюся площадь, но F с радиусом 1 остаётся равным 1. Это важное
  ограничение допуска: мелкий дефект может целиком находиться внутри tolerance band.
- `empty`: объект полностью потерян, обе метрики равны 0.

Высокий F сам по себе не доказывает, что площадь объекта восстановлена точно,
а высокий J может скрыть систематически неровный контур большого объекта.

## 3. SAM 2 для видео: как изображение превращается в трек


### Как SAM 2 обрабатывает последовательность

Видео читается последовательно. Для очередного кадра происходят четыре шага:

1. **Image encoder** вычисляет признаки текущего кадра.
2. **Memory attention** сопоставляет их с spatial memories и object pointers из прошлых кадров.
3. **Mask decoder** строит текущую маску, учитывая память и новый prompt, если он есть.
4. **Memory encoder** превращает предсказание в запись для следующих кадров.

Таким образом, SAM 2 одновременно похож на интерактивный segmenter и tracker. Плюс памяти — сохранение
идентичности при движении и частичной окклюзии. Минус — ошибочное предсказание тоже может попасть в память
и повлиять на будущее.

### Почему говорим о трёх гипотезах

В разделе 1 **image predictor** вернул три маски для одного неоднозначного клика. Это варианты только
текущего изображения. **Video predictor не получает автоматически выбранную картинку из того опыта**:
ему нужно явно передать point, box либо mask prompt, после чего он строит одну маску объекта и переносит
её во времени.

### Что сравниваем

Все три опыта используют один checkpoint и одни 80 кадров; меняется только информация о кадре 0:

- `point` — один положительный клик: самый быстрый, но семантически неоднозначный ввод;
- `box` — рамка полного объекта: сообщает охват, но не точную границу;
- `first-mask` — точная GT-маска кадра 0: стандартный semi-supervised VOS-протокол.


In [ ]:
# Idempotent cleanup: this cell may be rerun after a failed video experiment.
globals().pop("image_predictor", None)
globals().pop("image_model", None)
gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
VIDEO_OVERRIDES = [
    "++model.add_all_frames_to_correct_as_cond=true",
]
video_predictor = build_sam2_video_predictor(
    MODEL_CFG, str(CHECKPOINT), device=device,
    hydra_overrides_extra=VIDEO_OVERRIDES,
    apply_postprocessing=False)

def collect_masks(outputs, obj_id=1):
    # propagate_in_video возвращает generator из (frame_idx, object_ids, logits).
    # Здесь выбираем нужный obj_id, переносим logits на CPU и применяем порог 0.
    result = {}
    for frame_idx, object_ids, mask_logits in outputs:
        ids = [int(value) for value in object_ids]
        object_index = ids.index(obj_id)
        logits = mask_logits[object_index].detach().float().cpu().numpy()
        result[int(frame_idx)] = np.squeeze(logits > 0).astype(bool)
    return result

def run_sam2_experiment(*, first_points=None, first_labels=None, first_box=None,
                        first_mask=None, corrections=()):
    # Новый state изолирует этот эксперимент от масок и памяти предыдущего запуска.
    # offload_video_to_cpu экономит VRAM T4: кадры хранятся в RAM и подаются по мере надобности.
    state = video_predictor.init_state(video_path=str(FRAME_DIR), offload_video_to_cpu=True)
    video_predictor.reset_state(state)
    # Mask prompt наиболее информативен. Если его нет, используем точки и/или box.
    if first_mask is not None:
        video_predictor.add_new_mask(state, frame_idx=0, obj_id=1, mask=np.asarray(first_mask, bool))
    else:
        video_predictor.add_new_points_or_box(
            state, frame_idx=0, obj_id=1,
            points=None if first_points is None else np.asarray(first_points, np.float32),
            labels=None if first_labels is None else np.asarray(first_labels, np.int32),
            box=None if first_box is None else np.asarray(first_box, np.float32))
    # Первый forward pass переносит начальный prompt от кадра 0 до конца ролика.
    by_frame = collect_masks(video_predictor.propagate_in_video(state, start_frame_idx=0))
    for correction in corrections:
        frame_idx = int(correction["frame_idx"])
        video_predictor.add_new_points_or_box(
            state, frame_idx=frame_idx, obj_id=1,
            points=np.asarray(correction["points"], np.float32),
            labels=np.asarray(correction["labels"], np.int32))
        # One pass updates the future, the other updates frames before the click.
        forward = collect_masks(video_predictor.propagate_in_video(
            state, start_frame_idx=frame_idx,
            max_frame_num_to_track=N_FRAMES-frame_idx, reverse=False))
        reverse = collect_masks(video_predictor.propagate_in_video(
            state, start_frame_idx=frame_idx,
            max_frame_num_to_track=frame_idx+1, reverse=True))
        by_frame.update(forward); by_frame.update(reverse)
    missing = sorted(set(range(N_FRAMES))-set(by_frame))
    if missing:
        raise RuntimeError(f"SAM 2 did not return frames: {missing[:10]}")
    return state, [by_frame[index] for index in range(N_FRAMES)]

def sequence_metrics(masks, targets, start=1):
    # Кадр start=0 был бы prompt-кадром и искусственно завышал оценку, поэтому начинаем с 1.
    j = np.asarray([jaccard(m,g) for m,g in zip(masks[start:], targets[start:])])
    f = np.asarray([boundary_f(m,g) for m,g in zip(masks[start:], targets[start:])])
    return {"J": float(j.mean()), "F": float(f.mean()), "J&F": float((j.mean()+f.mean())/2),
            "J_per_frame": j, "F_per_frame": f,
            "frame_indices": np.arange(start, len(masks))}

In [ ]:
# Запускаем три независимых состояния
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    point_state, point_masks = run_sam2_experiment(
        first_points=[old_click], first_labels=[1])
    del point_state  # state больше не нужен; оставляем только готовые bool-маски.

    box_state, box_masks = run_sam2_experiment(first_box=box0)
    del box_state

    # Для честного semi-supervised протокола передаём только GT кадра 0.
    mask_state, mask_prompt_masks = run_sam2_experiment(first_mask=gts[0])

# Метрики всех методов вычисляются одинаково и не включают prompt frame.
sam_results = {
    "SAM2 point": sequence_metrics(point_masks, gts),
    "SAM2 box": sequence_metrics(box_masks, gts),
    "SAM2 first-mask": sequence_metrics(mask_prompt_masks, gts),
}
print(f"{'method':20s} {'J':>7s} {'F':>7s} {'J&F':>7s}")
for name, result in sam_results.items():
    print(f"{name:20s} {result['J']:7.3f} {result['F']:7.3f} {result['J&F']:7.3f}")

# Строки — тип начального prompt; столбцы — одинаковые моменты времени.
fig, axes = plt.subplots(3,4,figsize=(16,11))
for row, (name, masks) in enumerate([
        ("point", point_masks), ("box", box_masks), ("first mask", mask_prompt_masks)]):
    for ax, index in zip(axes[row], [0,25,50,79]):
        ax.imshow(overlay(frames[index], masks[index]))
        ax.set_title(f"{name}, frame {index}")
        ax.axis("off")
plt.tight_layout(); plt.show()

# Видео сохраняем только для сильнейшего протокола
sam2_video = save_video("sam2_first_mask.mp4",
                        [overlay(f,m) for f,m in zip(frames,mask_prompt_masks)])
display(Video(str(sam2_video), embed=True, width=720))

In [ ]:
# Покадровый J показывает не только среднее качество, но и моменты потери объекта.
plt.figure(figsize=(14,5))
for name, result in sam_results.items():
    plt.plot(result["frame_indices"], result["J_per_frame"], label=name)
plt.ylim(0,1.03)
plt.xlabel("номер кадра")
plt.ylabel("J: совпадение площади")
plt.title("Устойчивость трёх способов инициализации во времени")
plt.grid(alpha=.25); plt.legend(); plt.show()

# Выбираем лучший протокол по среднему J&F, а не по одному удачному кадру.
best_sam_name = max(sam_results, key=lambda name: sam_results[name]["J&F"])
print("Лучший SAM 2 протокол:", best_sam_name)

In [ ]:
def run_first_mask_sequence(frame_dir, targets):
    '''Одинаковый semi-supervised SAM 2 protocol для последовательности произвольной длины.'''
    state = video_predictor.init_state(
        video_path=str(frame_dir),
        offload_video_to_cpu=True,  # экономим VRAM и оставляем место модели
    )
    video_predictor.reset_state(state)

    # Единственный вход от человека — точная маска объекта на первом кадре.
    video_predictor.add_new_mask(
        state, frame_idx=0, obj_id=1, mask=np.asarray(targets[0], bool))

    predicted = collect_masks(video_predictor.propagate_in_video(state, start_frame_idx=0))
    expected = len(targets)
    assert set(predicted) == set(range(expected)), "Получены не все кадры"
    masks = [predicted[index] for index in range(expected)]
    return state, masks, sequence_metrics(masks, targets)

# Меняется только видео; checkpoint, protocol и метрики остаются теми же.
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    libby_state, libby_video_masks, libby_sam_result = run_first_mask_sequence(
        LIBBY_FRAME_DIR, libby_gts)
    del libby_state
    swan_state, swan_video_masks, swan_sam_result = run_first_mask_sequence(
        SWAN_FRAME_DIR, swan_gts)
    del swan_state

cross_scene_sam = {
    "bmx-trees": sam_results["SAM2 first-mask"],
    "libby": libby_sam_result,
    "blackswan": swan_sam_result,
}
print(f"{'sequence':14s} {'J':>7s} {'F':>7s} {'J&F':>7s}")
for name, result in cross_scene_sam.items():
    print(f"{name:14s} {result['J']:7.3f} {result['F']:7.3f} {result['J&F']:7.3f}")

show_frames(libby_frames, LIBBY_KEY_FRAMES, libby_video_masks,
            title="SAM 2 first-mask: собака libby")
show_frames(swan_frames, SWAN_KEY_FRAMES, swan_video_masks,
            title="SAM 2 first-mask: blackswan")

## 4. Исправляющий кадр: интерактивное восстановление трека

**Исправляющий кадр (correction/prompted frame)** — кадр, на котором пользователь видит ошибку трекера
и добавляет новые positive/negative clicks. Decoder объединяет клики с предыдущими mask logits, а
уточнённая маска становится новой надёжной записью памяти.

Алгоритм упражнения:

1. среди кадров 1–79 найти минимальный J;
2. разложить ошибку на TP, FN и FP;
3. поставить positive click глубоко в FN и negative click глубоко в FP;
4. повторить propagation вперёд и назад от prompted frame;
5. сравнить не только сам исправленный кадр, но и средний J&F всей последовательности.


In [ ]:
def error_clicks(prediction, target, max_points=2):
    '''Предлагает не более одной positive FN-точки и одной negative FP-точки.'''
    points, labels = [], []
    error_regions = [
        (target & ~prediction, 1),  # FN: объект пропущен — positive click
        (prediction & ~target, 0),  # FP: захвачен фон — negative click
    ]
    for region, label in error_regions:
        if region.any() and len(points) < max_points:
            # deepest_click снижает риск поставить точку на спорной границе.
            points.append(deepest_click(region))
            labels.append(label)
    return np.asarray(points, np.float32), np.asarray(labels, np.int32)

# Baseline — first-mask protocol.
base = sam_results["SAM2 first-mask"]

# argmin возвращает позицию внутри массива J_per_frame; frame_indices переводит её в номер кадра.
worst_frame = int(base["frame_indices"][np.argmin(base["J_per_frame"])])
points, labels = error_clicks(
    mask_prompt_masks[worst_frame], gts[worst_frame], max_points=2)

# RGB-карта ошибки делает смысл кликов видимым до повторного запуска модели.
diagnostic = np.zeros_like(frames[worst_frame])
diagnostic[mask_prompt_masks[worst_frame] & gts[worst_frame]] = (0,200,0)   # TP
diagnostic[gts[worst_frame] & ~mask_prompt_masks[worst_frame]] = (30,144,255)  # FN
diagnostic[mask_prompt_masks[worst_frame] & ~gts[worst_frame]] = (255,60,60)   # FP

fig, axes = plt.subplots(1,3,figsize=(16,5))
for ax, image, title in zip(
        axes,
        [frames[worst_frame], overlay(frames[worst_frame], mask_prompt_masks[worst_frame]), diagnostic],
        [f"frame {worst_frame}", "предсказание до кликов", "TP / FN / FP"]):
    ax.imshow(image); ax.set_title(title); ax.axis("off")
axes[-1].scatter(points[:,0], points[:,1],
                 c=["lime" if label==1 else "red" for label in labels],
                 s=140, marker="*")
plt.show()

# Correction хранит всё взаимодействие: кадр, координаты и типы точек.
correction = {"frame_idx": worst_frame, "points": points, "labels": labels}
with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
    refined_state, refined_masks = run_sam2_experiment(
        first_mask=gts[0], corrections=[correction])

refined_result = sequence_metrics(refined_masks, gts)
print("before:", {k:round(base[k],3) for k in ("J","F","J&F")})
print("after: ", {k:round(refined_result[k],3) for k in ("J","F","J&F")})

In [ ]:
# Если кривые расходятся только рядом с вертикальной линией, эффект локален.
# Долгое расхождение после линии означает, что исправление изменило будущую memory trajectory.
plt.figure(figsize=(14,5))
plt.plot(base["frame_indices"], base["J_per_frame"], label="до correction")
plt.plot(refined_result["frame_indices"], refined_result["J_per_frame"], label="после correction")
plt.axvline(worst_frame, color="black", ls="--", label="prompted frame")
plt.ylim(0,1.03)
plt.xlabel("номер кадра"); plt.ylabel("J")
plt.title("Локальный и временной эффект исправляющего кадра")
plt.grid(alpha=.25); plt.legend(); plt.show()

## 5. XMem: альтернативная архитектура памяти

**XMem** — метод semi-supervised video object segmentation, созданный специально для длинных видео.
Как и в нашем честном SAM 2 protocol, он получает маску объекта на первом кадре и должен продолжить её
на остальные кадры без доступа к будущему GT.

### Три уровня памяти

- **Sensory memory** обновляется каждый кадр и хранит краткосрочную пространственную информацию.
- **Working memory** содержит несколько недавних информативных кадров с высоким разрешением признаков.
- **Long-term memory** сжимает старые записи в прототипы, ограничивая рост памяти на длинном ролике.

Это отличается от SAM 2: XMem не является универсальным promptable image segmenter и не начинает работу
с произвольного point/box prompt. Его сильная сторона — перенос уже заданной точной маски через время.

### Что сравниваем

SAM 2 first-mask и XMem получают **одну и ту же GT-маску только кадра 0**. `VideoReader` XMem при
`use_all_mask=False` загружает только первую маску; остальные PNG служат списком имён для сохранения и
метриками после завершения инференса.

| Параметр | Смысл |
|---|---|
| `mem_every=5` | добавлять кадр в working memory примерно раз в пять шагов |
| `top_k=30` | читать 30 наиболее похожих memory locations |
| `max_mid_term_frames=10` | верхняя граница числа кадров рабочей памяти до консолидации |
| `num_prototypes=128` | число прототипов, которыми представляется long-term memory |
| исходное разрешение | generic loader v1.0 использует исходные DAVIS 480p кадры |

In [ ]:
# SAM 2 и его states больше не нужны до click-budget section.
del refined_state, mask_state, video_predictor
gc.collect()
torch.cuda.empty_cache()

XMEM_ROOT = Path("/content/XMem") if Path("/content").exists() else Path("XMem")
XMEM_MODEL = Path("/content/XMem.pth") if Path("/content").exists() else Path("XMem.pth")
XMEM_COMMIT = "ec776ea"  # официальный release v1.0

# Клонируем и устанавливаем только при первом запуске runtime.
if not XMEM_ROOT.exists():
    run(["git", "clone", "--quiet", "https://github.com/hkchengrex/XMem.git", str(XMEM_ROOT)])
run(["git", "-C", str(XMEM_ROOT), "checkout", "--quiet", XMEM_COMMIT])
run([sys.executable, "-m", "pip", "install", "-q",
     "progressbar2", "hickle", "gdown", "gitpython"])

# Checkpoint XMem отдельный от SAM 2: архитектуры и веса моделей не совместимы.
if not XMEM_MODEL.exists():
    run(["curl", "-L", "--fail", "--retry", "3", "-o", str(XMEM_MODEL),
         "https://github.com/hkchengrex/XMem/releases/download/v1.0/XMem.pth"])
print("XMem checkpoint:", round(XMEM_MODEL.stat().st_size/2**20,1), "MB")

In [ ]:
# Официальный generic loader ожидает JPEGImages/<video> и Annotations/<video>.
XMEM_DATA = Path("/content/xmem_bmx") if Path("/content").exists() else Path("xmem_bmx")
XMEM_OUTPUT = Path("/content/xmem_output") if Path("/content").exists() else Path("xmem_output")

# Создаём требуемую структуру и копируем кадры/аннотации под именем последовательности.
(XMEM_DATA/"JPEGImages"/"bmx-trees").mkdir(parents=True, exist_ok=True)
(XMEM_DATA/"Annotations"/"bmx-trees").mkdir(parents=True, exist_ok=True)
for source_path in FRAME_DIR.glob("*.jpg"):
    shutil.copy2(source_path, XMEM_DATA/"JPEGImages"/"bmx-trees"/source_path.name)
for source_path in MASK_DIR.glob("*.png"):
    shutil.copy2(source_path, XMEM_DATA/"Annotations"/"bmx-trees"/source_path.name)

# eval.py сам читает только первую mask как вход и сохраняет prediction каждого кадра.
run([sys.executable, "eval.py", "--model", str(XMEM_MODEL), "--dataset", "G",
     "--generic_path", str(XMEM_DATA), "--output", str(XMEM_OUTPUT),
     "--mem_every", "5", "--top_k", "30"], cwd=str(XMEM_ROOT))

# Загружаем PNG-предсказания, проверяем полноту и считаем те же J/F, что для SAM 2.
xmem_paths = sorted((XMEM_OUTPUT/"bmx-trees").glob("*.png"))
assert len(xmem_paths) == N_FRAMES, f"Expected {N_FRAMES} XMem masks, got {len(xmem_paths)}"
xmem_masks = [np.asarray(Image.open(path)) > 0 for path in xmem_paths]
xmem_result = sequence_metrics(xmem_masks, gts)
print("XMem bmx-trees:", {k:round(xmem_result[k],3) for k in ("J","F","J&F")})

# В общей таблице одинаковые столбцы и одинаковое исключение prompt frame.
comparison = {**sam_results, "SAM2 corrected": refined_result, "XMem first-mask": xmem_result}
print(f"{'method':20s} {'J':>7s} {'F':>7s} {'J&F':>7s}")
for name, result in comparison.items():
    print(f"{name:20s} {result['J']:7.3f} {result['F']:7.3f} {result['J&F']:7.3f}")

show_frames(frames, KEY_FRAMES, xmem_masks, title="XMem: предсказания на bmx-trees")
xmem_video = save_video("xmem_bmx_trees.mp4",
                        [overlay(f,m) for f,m in zip(frames,xmem_masks)])
display(Video(str(xmem_video), embed=True, width=720))

In [ ]:
def prepare_xmem_sequence(root, name, frame_dir, mask_dir):
    '''Копирует одну последовательность в структуру generic loader XMem.'''
    image_target = root/"JPEGImages"/name
    mask_target = root/"Annotations"/name
    image_target.mkdir(parents=True, exist_ok=True)
    mask_target.mkdir(parents=True, exist_ok=True)
    for path in frame_dir.glob("*.jpg"):
        shutil.copy2(path, image_target/path.name)
    for path in mask_dir.glob("*.png"):
        shutil.copy2(path, mask_target/path.name)

XMEM_EXTRA_DATA = Path("/content/xmem_extra")
XMEM_EXTRA_OUTPUT = Path("/content/xmem_extra_output")
prepare_xmem_sequence(XMEM_EXTRA_DATA, "libby", LIBBY_FRAME_DIR, LIBBY_MASK_DIR)
prepare_xmem_sequence(XMEM_EXTRA_DATA, "blackswan", SWAN_FRAME_DIR, SWAN_MASK_DIR)

# Один вызов eval.py последовательно обработает обе папки теми же параметрами памяти.
run([sys.executable, "eval.py", "--model", str(XMEM_MODEL), "--dataset", "G",
     "--generic_path", str(XMEM_EXTRA_DATA), "--output", str(XMEM_EXTRA_OUTPUT),
     "--mem_every", "5", "--top_k", "30"], cwd=str(XMEM_ROOT))

xmem_extra_results = {}
xmem_extra_masks = {}
for name, targets in [("libby", libby_gts), ("blackswan", swan_gts)]:
    paths = sorted((XMEM_EXTRA_OUTPUT/name).glob("*.png"))
    assert len(paths) == len(targets), f"{name}: ожидалось {len(targets)} масок, получено {len(paths)}"
    masks = [np.asarray(Image.open(path)) > 0 for path in paths]
    xmem_extra_masks[name] = masks
    xmem_extra_results[name] = sequence_metrics(masks, targets)

print(f"{'sequence':14s} {'SAM2 J&F':>10s} {'XMem J&F':>10s}")
for name, sam_result in [("libby", libby_sam_result), ("blackswan", swan_sam_result)]:
    print(f"{name:14s} {sam_result['J&F']:10.3f} {xmem_extra_results[name]['J&F']:10.3f}")

show_frames(libby_frames, LIBBY_KEY_FRAMES, xmem_extra_masks["libby"],
            title="XMem first-mask: собака libby")
show_frames(swan_frames, SWAN_KEY_FRAMES, xmem_extra_masks["blackswan"],
            title="XMem first-mask: blackswan")

## 6. Click-budget challenge с исполняемым поиском

В интерактивной разметке время человека дороже дополнительного forward pass. Поэтому задача формулируется
не как «сделать маску любой ценой», а как **максимизировать J&F при фиксированном числе действий**.

Условия:

- точная first-frame mask считается исходным условием semi-supervised VOS и не входит в бюджет;
- разрешено не более двух correction clicks;
- один кандидат — один новый независимый запуск, иначе память предыдущего опыта исказит сравнение.

Полный перебор всех кадров и пикселей слишком дорог. Используем ограниченный oracle search:

1. ранжируем кадры baseline по низкому J;
2. выбираем четыре кадра на расстоянии хотя бы 8 кадров друг от друга;
3. в каждом предлагаем глубокий positive FN и negative FP click;
4. реально выполняем forward/reverse propagation и считаем средний J&F;
5. выбираем лучший результат, **включая возможность почти не улучшить baseline**.


In [ ]:
gc.collect(); torch.cuda.empty_cache()
video_predictor = build_sam2_video_predictor(
    MODEL_CFG, str(CHECKPOINT), device=device,
    hydra_overrides_extra=VIDEO_OVERRIDES,
    apply_postprocessing=False)

def spaced_worst_frames(metric, count=4, min_gap=8):
    '''Выбирает плохие, но не соседние кадры: соседние часто представляют один и тот же сбой.'''
    ranked = metric["frame_indices"][np.argsort(metric["J_per_frame"])]
    chosen = []
    for frame_idx in ranked:
        if all(abs(int(frame_idx)-old) >= min_gap for old in chosen):
            chosen.append(int(frame_idx))
        if len(chosen) == count:
            break
    return chosen

candidate_frames = spaced_worst_frames(base)
challenge_rows = []

for frame_idx in candidate_frames:
    # Две oracle-точки строятся по ошибкам baseline, а не по результату другого кандидата.
    candidate_points, candidate_labels = error_clicks(
        mask_prompt_masks[frame_idx], gts[frame_idx], max_points=2)
    candidate = {"frame_idx": frame_idx,
                 "points": candidate_points,
                 "labels": candidate_labels}

    # Каждый кандидат начинает с одной и той же first-frame mask и чистой memory trajectory.
    with torch.inference_mode(), torch.autocast("cuda", dtype=torch.float16):
        candidate_state, candidate_masks = run_sam2_experiment(
            first_mask=gts[0], corrections=[candidate])
    candidate_result = sequence_metrics(candidate_masks, gts)

    challenge_rows.append({
        "frame": frame_idx,
        "points": candidate_points.tolist(),
        "labels": candidate_labels.tolist(),
        "result": candidate_result,
        "masks": candidate_masks,
    })
    del candidate_state
    print(f"frame={frame_idx:2d}, clicks={len(candidate_points)}, "
          f"J&F={candidate_result['J&F']:.3f}")

# Победитель выбирается по метрике всей последовательности, а не по J prompted frame.
winner = max(challenge_rows, key=lambda row: row["result"]["J&F"])
assert len(winner["points"]) <= 2
print("WINNER:", {"frame":winner["frame"], "points":winner["points"],
                  "labels":winner["labels"], "J&F":round(winner["result"]["J&F"],3)})

In [ ]:
# На одном графике сравниваем площадь маски во времени:
# baseline, лучший двухкликовый вариант и другую архитектуру памяти XMem.
plt.figure(figsize=(14,5))
plt.plot(base["frame_indices"], base["J_per_frame"], label="SAM2 first-mask baseline")
plt.plot(winner["result"]["frame_indices"], winner["result"]["J_per_frame"],
         label="SAM2 click-budget winner")
plt.plot(xmem_result["frame_indices"], xmem_result["J_per_frame"], label="XMem")
plt.axvline(winner["frame"], color="black", ls="--", alpha=.7,
            label="winner prompted frame")
plt.ylim(0,1.03)
plt.xlabel("номер кадра"); plt.ylabel("J")
plt.title("Что изменили два клика и где методы теряют объект")
plt.grid(alpha=.25); plt.legend(); plt.show()

winner_video = save_video(
    "sam2_click_budget_winner.mp4",
    [overlay(frame, mask) for frame, mask in zip(frames, winner["masks"])])
display(Video(str(winner_video), embed=True, width=720))